# Task B Run 29 -- frozen Gemma, classification head only

Gemma-4-12B reached 0.6792 macro-F1 with two-seed QLoRA in Run 21. This quick ablation asks whether the pretrained Gemma representation is already sufficient, or whether updating LoRA weights is responsible for most of the gain.

The Gemma backbone is loaded in 4-bit but kept completely frozen. No LoRA adapter is attached and only the six-class linear head is trained. The same prompt, class weighting, label smoothing, three epochs and fixed 530-row holdout are used. Two seeds are averaged.

This is evaluation-only: it writes holdout probabilities and a result summary, but no test ZIP. It should take roughly **45--90 minutes on T4 x2**, including model loading.

In [ ]:
import glob, json, os, pathlib, shutil, subprocess, sys, time
REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"
WORK = "/kaggle/working/hastika"

def git(*args):
    return subprocess.run(["git", "-C", WORK, *args], check=True, capture_output=True, text=True).stdout.strip()

remote = subprocess.run(["git", "ls-remote", "--exit-code", REPO, f"refs/heads/{BRANCH}"], check=True, capture_output=True, text=True, timeout=60).stdout.split()[0]
if os.path.isdir(WORK + "/.git"):
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)
head = git("rev-parse", "HEAD")
assert head == remote, f"clone is at {head[:8]} but {BRANCH} is at {remote[:8]}"
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret loaded")
except Exception:
    print("HF_TOKEN not found; Gemma access may fail")

import numpy as np
import pandas as pd
import torch
assert torch.cuda.is_available(), "select a GPU accelerator"
N_GPU = torch.cuda.device_count()
print("repo", git("log", "-1", "--oneline"))
print("gpus:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])

OUT = pathlib.Path("/kaggle/working/b29_frozen_head_outputs")
RUNS, LOGS = OUT / "runs", OUT / "logs"
for p in (RUNS, LOGS): p.mkdir(parents=True, exist_ok=True)
HF_CACHE = "/tmp/hf"
os.environ["HF_HUB_CACHE"] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import LABELS, fingerprint, paired_bootstrap, report
SPLIT = pathlib.Path(WORK) / "data/derived/task_b_combined_holdout"
train = pd.read_csv(SPLIT / "train.csv")
holdout = pd.read_csv(SPLIT / "holdout.csv")
assert (len(train), len(holdout)) == (3002, 530), (len(train), len(holdout))
HOLDOUT_FP = fingerprint(holdout["id"])
assert HOLDOUT_FP == "f85f4f049b", HOLDOUT_FP
y_ho = holdout["Hate Category"].map(LABELS.index).to_numpy()
MODEL = "google/gemma-4-12B"
SEEDS = (42, 43)
print("train", len(train), "holdout", len(holdout), "fingerprint", HOLDOUT_FP)

In [ ]:
LLM_PY = "/tmp/llmenv/bin/python"
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", "/tmp/llmenv"], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" "bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf', shell=True, check=True)
subprocess.run([LLM_PY, "-c", "import torch, transformers, peft, bitsandbytes; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__)"], check=True)
from huggingface_hub import snapshot_download
snapshot_download(MODEL, cache_dir=HF_CACHE, token=os.environ.get("HF_TOKEN") or None, allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*", "*.jinja"])

def job(seed):
    name = f"gemma_frozen_head_s{seed}"
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_classifier --model {MODEL} "
                    f"--train {SPLIT / 'train.csv'} --eval {SPLIT / 'holdout.csv'} "
                    f"--predict {SPLIT / 'holdout.csv'} --out {RUNS / name} "
                    f"--seed {seed} --epochs 3 --head-only --bs 16 --grad-accum 1 "
                    f"--eval-bs 32 --head-lr 5e-4 --cache {HF_CACHE}")}

jobs = [job(s) for s in SEEDS]
print("queued:", [j["name"] for j in jobs])

In [ ]:
codes = run_queue(jobs, N_GPU, raise_on_fail=False, stop_at=time.time() + 10 * 3600)
print("exit codes:", codes)

def load(path):
    p = pathlib.Path(path) / "holdout_probs.npy"
    return np.load(p) if p.exists() else None

head_probs = {s: load(RUNS / f"gemma_frozen_head_s{s}") for s in SEEDS}
head_probs = {s: p for s, p in head_probs.items() if p is not None}
assert head_probs, "no frozen-head probabilities were produced; inspect the logs"
for s, p in head_probs.items(): report(y_ho, p, f"frozen head seed {s}")
head_mean = np.mean(list(head_probs.values()), 0)
head_score = report(y_ho, head_mean, "frozen head mean")

# Run 21's probabilities are optional. If attached, calculate a paired comparison.
baseline = {}
found = sorted(glob.glob("/kaggle/input/**/b21_outputs/result.json", recursive=True))
if found:
    b21 = pathlib.Path(found[0]).parent
    r21 = json.load(open(b21 / "result.json"))
    if r21.get("holdout_fingerprint") == HOLDOUT_FP:
        for s in SEEDS:
            p = load(b21 / "runs" / f"gemma-4-12b_ho_s{s}")
            if p is not None: baseline[s] = p
if baseline:
    baseline_mean = np.mean(list(baseline.values()), 0)
    base_score = report(y_ho, baseline_mean, "Run 21 QLoRA mean")
    comparison = paired_bootstrap(y_ho, head_mean, baseline_mean, n=5000, seed=2600)
    print(f"frozen head - QLoRA: {comparison['diff']:+.4f}; CI [{comparison['ci95'][0]:+.4f}, {comparison['ci95'][1]:+.4f}]; P(better) {comparison['p_better']:.2f}")
else:
    base_score, comparison = None, None
    print("Run 21 output was not attached; compare the frozen-head score with the recorded 0.6792 baseline.")

In [ ]:
record = {
    "commit": head, "model": MODEL, "holdout_rows": len(holdout),
    "holdout_fingerprint": HOLDOUT_FP, "train_rows": len(train),
    "head_only": True, "classifier_epochs": 3, "seeds": list(SEEDS),
    "frozen_head_macro_f1": head_score, "run21_qlora_macro_f1": base_score,
    "frozen_head_minus_qlora": comparison, "exit_codes": codes,
    "completed_seeds": sorted(map(int, head_probs)),
}
json.dump(record, open(OUT / "result.json", "w"), indent=2)
print(json.dumps(record, indent=2))
print("No submission ZIP is created by this diagnostic.")